# Robinhood Day Trading Bot — Dry Run Test

This notebook runs the trading bot in **DRY RUN mode** — it will:
- ✅ Log into your Robinhood account
- ✅ Scan for stocks with momentum signals
- ✅ Show you exactly what trades it *would* make
- ❌ NOT place any real orders

Run each cell from top to bottom. Click the play button ▶ on the left of each cell.

In [ ]:
# STEP 1 — Install dependencies (takes ~1 minute)
!pip install robin_stocks yfinance pandas_ta pandas numpy pytz -q

In [ ]:
# STEP 2 — Enter your Robinhood credentials
# These are only stored in this session and never saved anywhere
import os
import getpass

os.environ['ROBINHOOD_USERNAME'] = input('Enter your Robinhood email: ')
os.environ['ROBINHOOD_PASSWORD'] = getpass.getpass('Enter your Robinhood password (hidden): ')
print('Credentials set.')

In [ ]:
# STEP 3 — Download the trading script
import os, sys, urllib.request

# Download trader.py directly from GitHub
url = 'https://raw.githubusercontent.com/nrienks23-sketch/Robinhood/claude/dreamy-noether-4u7oj3/trader.py'
dest = '/content/trader.py'

urllib.request.urlretrieve(url, dest)

if not os.path.exists(dest):
    raise Exception('Download failed — check your internet connection.')

sys.path.insert(0, '/content')
os.chdir('/content')
print('Script loaded. trader.py is ready.')

In [ ]:
# STEP 4 — Run a single scan in dry run mode
# This will log into Robinhood (may ask for 2FA code),
# scan for signals, and print what trades it would make.
# It will NOT loop — just one scan so you can see results quickly.

import sys, os

# Make sure Python can find trader.py
for path in ['/content/Robinhood', 'Robinhood', '.']:
    if os.path.exists(os.path.join(path, 'trader.py')):
        if path not in sys.path:
            sys.path.insert(0, path)
        os.chdir(path)
        break

import importlib
import trader
importlib.reload(trader)  # ensures fresh load

# Confirm dry run is on
assert trader.DRY_RUN == True, 'DRY_RUN must be True for this test!'

# Login
logged_in = trader.rh_login()

if logged_in:
    print('\n✅ Logged into Robinhood successfully!\n')

    # Load any existing positions
    positions = trader.load_positions()

    # Run one scan
    print('Scanning for entry signals... (this takes a few minutes)\n')
    entries = trader.scan_for_entries(positions)

    print('\n' + '='*60)
    if entries:
        print(f'  {len(entries)} trade signal(s) found:')
        for ticker, price in entries:
            shares = int(trader.POSITION_SIZE_USD // price)
            print(f'  → Would BUY {shares} shares of {ticker} @ ${price:.2f} (~${shares*price:.2f})')
    else:
        print('  No signals found this scan. Try again during market hours (9:30am-4pm ET).')
    print('='*60)
else:
    print('❌ Login failed. Check your username/password and try again.')

In [ ]:
# OPTIONAL — Run the full loop for 10 minutes (dry run)
# This will scan every 30 seconds for 10 minutes so you can watch it work.
# Press the STOP button (■) in the toolbar to stop it early.

import time

positions = trader.load_positions()
end_time = time.time() + 600  # 10 minutes
scan_count = 0

while time.time() < end_time:
    scan_count += 1
    print(f'\n--- Scan #{scan_count} ---')
    trader.manage_positions(positions)
    entries = trader.scan_for_entries(positions)
    for ticker, price in entries:
        trader.enter_position(ticker, price, positions)
    trader.print_summary(positions, entries)
    print(f'Waiting 30 seconds...')
    time.sleep(30)

print('10-minute test complete.')